In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import kagglehub

# Prompts for your Kaggle username and key from kaggle.json
kagglehub.login()



Kaggle credentials set.
Kaggle credentials successfully validated.


In [3]:
from pathlib import Path

# Colab path
DATA_DIR = Path("/content/data/hm")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Download H&M
path = kagglehub.competition_download(
    "h-and-m-personalized-fashion-recommendations",
    output_dir=str(DATA_DIR)
)

print("Downloaded to:", path)

100%|██████████| 28.7G/28.7G [23:41<00:00, 21.7MB/s]

Extracting files...


Downloaded to: /content/data/hm


In [4]:
from pathlib import Path
import shutil

# Define source and destination directories
SRC_DIR = Path("/content/data/hm")
DST_DIR = Path(
    "/content/drive/MyDrive/projects/e_commerce_recsys/data/original data"
)

# Create the destination directory if it does not exist
DST_DIR.mkdir(parents=True, exist_ok=True)

# Define files to copy
FILES = [
    "sample_submission.csv",
    "transactions_train.csv",
    "customers.csv",
    "articles.csv",
]

# Copy files and verify their sizes
for filename in FILES:
    src = SRC_DIR / filename
    dst = DST_DIR / filename

    if not src.is_file():
        print(f"[ERROR] Source file not found: {src}")
        continue

    shutil.copy2(src, dst)

    if dst.is_file() and src.stat().st_size == dst.stat().st_size:
        size_mb = dst.stat().st_size / (1024 ** 2)
        print(f"[SUCCESS] {filename} ({size_mb:.2f} MB)")
    else:
        print(f"[ERROR] Verification failed: {filename}")

print("\nFile copy process completed.")

[SUCCESS] sample_submission.csv (257.76 MB)
[SUCCESS] transactions_train.csv (3326.42 MB)
[SUCCESS] customers.csv (197.54 MB)
[SUCCESS] articles.csv (34.45 MB)

File copy process completed.


# H&M Image Features — CLIP ViT-B/32 (512D)

**Purpose:** Produce one reproducible 512-dimensional CLIP image embedding per successfully decoded H&M article image, then build an exact cosine-similarity FAISS index.

**Input:** `/content/data/hm/articles.csv` and `/content/data/hm/images/`.

**Published output (after validation):** `/content/drive/MyDrive/projects/e_commerce_recsys/data/image_features/`

- `image_embeddings.npy`: float32, L2-normalized embedding rows, shape `(N_success, 512)`.
- `image_article_ids.parquet`: `embedding_row`, ten-digit **string** `article_id`, and relative `image_path` in identical row order.
- `image_faiss.index`: FAISS `IndexFlatIP` with vector position equal to `embedding_row`.
- Supplementary audit: `image_features_manifest.json` and `image_embedding_failures.parquet`.

**Mapping:** For `article_id = "0108775015"`, Kaggle stores the image as `images/010/0108775015.jpg` (three-digit directory prefix and zero-preserved ten-digit filename). The actual files are enumerated, not assumed to exist.

**EDA baseline:** 105,542 catalog articles; 105,100 locally present, uniquely mapped images; 442 catalog articles with no image; 99.5812% filename-level catalog coverage. Filename presence does not guarantee decodability. This notebook measures actual embedding success and explicitly excludes failed images.

This notebook uses pretrained CLIP only; it does **not** train a recommender or use future transactions. It performs image reads on Colab local storage and publishes only a few aggregate assets to Drive. Full catalog preprocessing can be expensive; enable a GPU in **Runtime → Change runtime type** before running.

The intermediate checkpoint is stored on Colab local disk. It supports rerunning interrupted cells within the **same runtime**, but does not survive runtime deletion. Do not clear the working directory until final assets have been verified on Google Drive.

## Cell 1 — Install dependencies

Run once per Colab runtime. Use the preinstalled PyTorch CUDA build; do not reinstall PyTorch.

In [5]:
%pip -q install "transformers>=4.46,<5" "faiss-cpu>=1.8" "pyarrow>=14" "Pillow>=10" "tqdm>=4.66"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


## Cell 2 — Configuration and runtime checks

In [6]:
from pathlib import Path
import json
import os
import re
import hashlib
import shutil
import platform
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
import faiss
import transformers
from PIL import Image, ImageOps
from tqdm.auto import tqdm

DATA_DIR = Path("/content/data/hm")
ARTICLES_CSV = DATA_DIR / "articles.csv"
IMAGE_DIR = DATA_DIR / "images"

WORK_DIR = Path("/content/hm_image_features_work")
STAGING_DIR = WORK_DIR / "validated"
DRIVE_DIR = Path("/content/drive/MyDrive/projects/e_commerce_recsys/data/image_features")

MODEL_ID = "openai/clip-vit-base-patch32"
EMBEDDING_DIM = 512
BATCH_SIZE = 64
NUM_WORKERS = 4
CHECKPOINT_EVERY_BATCHES = 50
RETRY_FAILED_IMAGES = False
STRICT_EDA_COUNTS = True
EXPECTED_ARTICLES = 105_542
EXPECTED_LOCAL_IMAGE_FILES = 105_100

assert ARTICLES_CSV.is_file(), f"Missing catalog: {ARTICLES_CSV}"
assert IMAGE_DIR.is_dir(), f"Missing image directory: {IMAGE_DIR}"
assert torch.cuda.is_available(), "GPU is required for this full-catalog run. Enable a Colab GPU runtime."

WORK_DIR.mkdir(parents=True, exist_ok=True)
STAGING_DIR.mkdir(parents=True, exist_ok=True)

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("FAISS:", faiss.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("Data:", DATA_DIR)
print("Local working directory:", WORK_DIR)
print("Drive publication directory:", DRIVE_DIR)
print("Local disk free (GiB):", round(shutil.disk_usage('/content').free / 1024**3, 2))

Python: 3.13.16
PyTorch: 2.11.0+cu130
Transformers: 4.57.6
FAISS: 1.15.1
GPU: NVIDIA L4
Data: /content/data/hm
Local working directory: /content/hm_image_features_work
Drive publication directory: /content/drive/MyDrive/projects/e_commerce_recsys/data/image_features
Local disk free (GiB): 199.2


## Cell 3 — Enumerate image files and validate article ID mappings

An H&M `article_id` is a **string** with ten digits. Do not cast it to an integer. The image inventory is built by scanning actual local files; expected paths are used only for validation. Refuse to proceed if the supplied dataset differs from the audited EDA baseline (unless deliberately disabling `STRICT_EDA_COUNTS`).

In [7]:
catalog = pd.read_csv(ARTICLES_CSV, usecols=["article_id"], dtype={"article_id": "string"})
assert catalog["article_id"].notna().all(), "Null article IDs detected."
assert catalog["article_id"].str.fullmatch(r"[0-9]{10}").all(), "Unexpected article ID format."
assert catalog["article_id"].is_unique, "Duplicate article IDs detected."
catalog_ids = set(catalog["article_id"].tolist())

records = []
invalid_files = []
for path in IMAGE_DIR.rglob("*"):
    if not path.is_file():
        continue
    rel = path.relative_to(IMAGE_DIR)
    image_id = path.stem
    valid = (
        path.suffix.lower() in {".jpg", ".jpeg"}
        and bool(re.fullmatch(r"[0-9]{10}", image_id))
        and len(rel.parts) == 2
        and rel.parts[0] == image_id[:3]
    )
    if not valid:
        invalid_files.append(str(rel))
        continue
    records.append({
        "article_id": image_id,
        "image_path": rel.as_posix(),
        "bytes": int(path.stat().st_size),
    })

inventory = pd.DataFrame(records, columns=["article_id", "image_path", "bytes"])
inventory = inventory.sort_values("article_id", kind="stable").reset_index(drop=True)
assert not invalid_files, f"Invalid image filenames or directory prefixes (first 10): {invalid_files[:10]}"
assert inventory["article_id"].is_unique, "Multiple images map to one article ID."
assert set(inventory["article_id"]).issubset(catalog_ids), "Some image IDs are absent from articles.csv."
assert (inventory["bytes"] > 0).all(), "Zero-byte image files detected."

if STRICT_EDA_COUNTS:
    assert len(catalog) == EXPECTED_ARTICLES, "Catalog count differs from Phase 0 EDA."
    assert len(inventory) == EXPECTED_LOCAL_IMAGE_FILES, "Image count differs from Phase 0 EDA."

catalog_missing_images = len(catalog_ids - set(inventory["article_id"]))
print(f"Catalog articles: {len(catalog):,}")
print(f"Valid local image files: {len(inventory):,}")
print(f"Catalog articles without local images: {catalog_missing_images:,}")
print(f"Local filename-level catalog coverage: {100 * len(inventory) / len(catalog):.4f}%")
assert len(inventory) > 0, "No valid image files were found."

inventory.to_parquet(WORK_DIR / "source_image_inventory.parquet", index=False)

image_ids = inventory["article_id"].tolist()
image_paths = [(IMAGE_DIR / rel) for rel in inventory["image_path"].tolist()]

fingerprint = hashlib.sha256()
fingerprint.update(MODEL_ID.encode("utf-8"))
fingerprint.update(ARTICLES_CSV.read_bytes())
for row in inventory.itertuples(index=False):
    fingerprint.update(f"{row.article_id}|{row.image_path}|{row.bytes}\n".encode("utf-8"))
source_fingerprint = fingerprint.hexdigest()
print("Source/model fingerprint:", source_fingerprint)

Catalog articles: 105,542
Valid local image files: 105,100
Catalog articles without local images: 442
Local filename-level catalog coverage: 99.5812%
Source/model fingerprint: 1565e66320cd4a1dff424151fe5b62725322f7c5a6c02b5a1c8c823dd31871f5


## Cell 4 — Initialize a resumable local checkpoint

The checkpoint stores one `float32` row per enumerated image (about 205 MiB), plus status values:

- `0`: pending
- `1`: embedding verified and saved
- `2`: image read/preprocessing failure
- `3`: non-finite/invalid embedding

Successful images will later be compacted into contiguous final rows. A changed catalog, image inventory or model ID cannot silently reuse an older checkpoint.

In [8]:
ALL_ROWS_PATH = WORK_DIR / "all_image_embeddings.npy"
STATUS_PATH = WORK_DIR / "image_status.npy"
STATE_PATH = WORK_DIR / "checkpoint_state.json"
FAILURE_LOG_PATH = WORK_DIR / "image_failures.jsonl"

checkpoint_exists = [p.exists() for p in (ALL_ROWS_PATH, STATUS_PATH, STATE_PATH)]
assert all(checkpoint_exists) or not any(checkpoint_exists), (
    "Incomplete checkpoint files detected. Inspect WORK_DIR before resetting."
)

if all(checkpoint_exists):
    state = json.loads(STATE_PATH.read_text())
    assert state["fingerprint"] == source_fingerprint, "Checkpoint/source fingerprint mismatch."
    assert state["model_id"] == MODEL_ID, "Checkpoint/model mismatch."
    assert state["embedding_dim"] == EMBEDDING_DIM, "Checkpoint dimension mismatch."
    assert state["total_images"] == len(inventory), "Checkpoint inventory-size mismatch."
    all_vectors = np.load(ALL_ROWS_PATH, mmap_mode="r+")
    status = np.load(STATUS_PATH, mmap_mode="r+")
    assert all_vectors.shape == (len(inventory), EMBEDDING_DIM)
    assert status.shape == (len(inventory),)
    print("Existing local checkpoint restored.")
else:
    all_vectors = np.lib.format.open_memmap(
        ALL_ROWS_PATH, mode="w+", dtype=np.float32,
        shape=(len(inventory), EMBEDDING_DIM)
    )
    status = np.lib.format.open_memmap(
        STATUS_PATH, mode="w+", dtype=np.uint8, shape=(len(inventory),)
    )
    status[:] = 0
    all_vectors.flush()
    status.flush()
    state = {
        "fingerprint": source_fingerprint,
        "model_id": MODEL_ID,
        "embedding_dim": EMBEDDING_DIM,
        "total_images": len(inventory),
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    STATE_PATH.write_text(json.dumps(state, indent=2))
    print("New local checkpoint created.")

if RETRY_FAILED_IMAGES:
    status[np.isin(status, [2, 3])] = 0
    status.flush()
    print("Previously failed images reset to pending.")

values, counts = np.unique(status, return_counts=True)
print("Checkpoint status counts:", dict(zip(values.astype(int).tolist(), counts.astype(int).tolist())))
print("Pending images:", int(np.count_nonzero(status == 0)))

New local checkpoint created.
Checkpoint status counts: {0: 105100}
Pending images: 105100


## Cell 5 — Load pretrained CLIP and validate output dimension

Preprocessing uses the checkpoint's official `CLIPImageProcessor`. Vision features are L2-normalized before being written. Mixed precision is used for GPU inference; all stored vectors are `float32`.

In [9]:
from transformers import CLIPModel, CLIPImageProcessor

DEVICE = torch.device("cuda")
processor = CLIPImageProcessor.from_pretrained(MODEL_ID)
model = CLIPModel.from_pretrained(MODEL_ID).to(DEVICE).eval()
assert model.config.projection_dim == EMBEDDING_DIM, (
    f"Expected {EMBEDDING_DIM} dimensions, got {model.config.projection_dim}."
)
model_revision = getattr(model.config, "_commit_hash", None)
print("Model:", MODEL_ID)
print("Model revision:", model_revision)
print("Projection dimensions:", model.config.projection_dim)
print("Image processor:", type(processor).__name__)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Model: openai/clip-vit-base-patch32
Model revision: 3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268
Projection dimensions: 512
Image processor: CLIPImageProcessor


## Cell 6 — Define robust image decoding and batched preprocessing

Failed images are recorded and excluded rather than represented by zero vectors. A worker failure does not terminate the entire catalog run. `NUM_WORKERS` can be reduced if a Colab runtime has limited CPU memory.

In [10]:
from torch.utils.data import Dataset, DataLoader

class HmImageDataset(Dataset):
    def __init__(self, paths, pending_indices, image_processor):
        self.paths = paths
        self.pending_indices = np.asarray(pending_indices, dtype=np.int64)
        self.image_processor = image_processor

    def __len__(self):
        return len(self.pending_indices)

    def __getitem__(self, position):
        global_index = int(self.pending_indices[position])
        path = self.paths[global_index]
        try:
            with Image.open(path) as img:
                rgb = ImageOps.exif_transpose(img).convert("RGB")
                tensor = self.image_processor(images=rgb, return_tensors="pt")["pixel_values"][0]
            return (global_index, tensor, None)
        except Exception as exc:
            error = f"{type(exc).__name__}: {str(exc)[:300]}"
            return (global_index, None, error)

def collate_images(rows):
    valid_indices = [index for index, tensor, error in rows if tensor is not None]
    valid_tensors = [tensor for index, tensor, error in rows if tensor is not None]
    failures = [(index, error) for index, tensor, error in rows if tensor is None]
    tensor_batch = torch.stack(valid_tensors) if valid_tensors else None
    return valid_indices, tensor_batch, failures

pending_indices = np.flatnonzero(status == 0)
dataset = HmImageDataset(image_paths, pending_indices, processor)
loader_options = {
    "batch_size": BATCH_SIZE,
    "shuffle": False,
    "num_workers": NUM_WORKERS,
    "collate_fn": collate_images,
    "pin_memory": True,
}
if NUM_WORKERS > 0:
    loader_options["persistent_workers"] = True
    loader_options["prefetch_factor"] = 2

loader = DataLoader(dataset, **loader_options)
print(f"Pending images: {len(dataset):,}")
print(f"Batches: {len(loader):,}")

Pending images: 105,100
Batches: 1,643


## Cell 7 — Encode all pending images with checkpointing

This is the time-consuming step. The progress bar advances per image batch. Checkpoints flush to **local disk** every `CHECKPOINT_EVERY_BATCHES` batches. If the cell is interrupted, rerun Cells 4–7 in the same Colab runtime to resume. If CUDA runs out of memory, reduce `BATCH_SIZE`, recreate the DataLoader (Cell 6), and rerun; processed rows remain preserved.

Keep the runtime active until the cell completes.

In [11]:
import torch.nn.functional as F

processed_now = 0
failures_now = 0

with open(FAILURE_LOG_PATH, "a", encoding="utf-8", buffering=1) as failure_log:
    with torch.inference_mode():
        for batch_number, (indices, pixel_values, failures) in enumerate(
            tqdm(loader, desc="CLIP image encoding", unit="batch"), start=1
        ):
            for index, error in failures:
                status[index] = 2
                failure_log.write(json.dumps({
                    "index": int(index), "article_id": image_ids[index],
                    "image_path": inventory.iloc[index]["image_path"],
                    "error": error,
                }) + "\n")
                failures_now += 1

            if pixel_values is not None:
                pixel_values = pixel_values.to(DEVICE, non_blocking=True)
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    features = model.get_image_features(pixel_values=pixel_values)

                if not isinstance(features, torch.Tensor):
                    if getattr(features, "pooler_output", None) is not None:
                        features = features.pooler_output
                    elif getattr(features, "image_embeds", None) is not None:
                        features = features.image_embeds
                    elif isinstance(features, (tuple, list)):
                        features = features[0]
                    else:
                        raise TypeError("Unsupported CLIP get_image_features output type.")

                features = features.float()
                assert features.ndim == 2 and features.shape[1] == EMBEDDING_DIM, (
                    f"Unexpected CLIP feature shape: {tuple(features.shape)}"
                )
                vector_norms = torch.linalg.vector_norm(features, dim=1)
                valid_mask = torch.isfinite(features).all(dim=1) & torch.isfinite(vector_norms) & (vector_norms > 1e-8)
                normalized = F.normalize(features, p=2, dim=1)
                valid_mask &= torch.isfinite(normalized).all(dim=1)
                normalized_np = normalized.cpu().numpy().astype(np.float32, copy=False)
                valid_mask_np = valid_mask.cpu().numpy()

                for row, global_index in enumerate(indices):
                    if valid_mask_np[row]:
                        all_vectors[global_index, :] = normalized_np[row]
                        status[global_index] = 1
                        processed_now += 1
                    else:
                        status[global_index] = 3
                        failure_log.write(json.dumps({
                            "index": int(global_index), "article_id": image_ids[global_index],
                            "image_path": inventory.iloc[global_index]["image_path"],
                            "error": "Invalid CLIP embedding (NaN, Inf, or zero norm)",
                        }) + "\n")
                        failures_now += 1

                del pixel_values, features, normalized

            if batch_number % CHECKPOINT_EVERY_BATCHES == 0:
                all_vectors.flush()
                status.flush()

all_vectors.flush()
status.flush()
assert int(np.count_nonzero(status == 0)) == 0, "Some images remain pending."
print("New successful embeddings:", processed_now)
print("New failures:", failures_now)
print("Total successful:", int(np.count_nonzero(status == 1)))
print("Total failed:", int(np.count_nonzero(np.isin(status, [2, 3]))))

CLIP image encoding:   0%|          | 0/1643 [00:00<?, ?batch/s]

New successful embeddings: 105100
New failures: 0
Total successful: 105100
Total failed: 0


## Cell 8 — Compact validated embeddings and write aligned Article ID mapping

Rows are compacted **only for successful images**; absent/undecodable articles are not assigned fake zero embeddings. The Parquet mapping is ordered exactly like the final `.npy` rows.

In [12]:
assert int(np.count_nonzero(status == 0)) == 0, "Encoding is incomplete."

success_indices = np.flatnonzero(status == 1)
failed_indices = np.flatnonzero(np.isin(status, [2, 3]))
N_SUCCESS = len(success_indices)
assert N_SUCCESS > 0, "There are no successful embeddings."

EMBEDDINGS_PATH = STAGING_DIR / "image_embeddings.npy"
IDS_PATH = STAGING_DIR / "image_article_ids.parquet"
FAILURES_PATH = STAGING_DIR / "image_embedding_failures.parquet"
INDEX_PATH = STAGING_DIR / "image_faiss.index"
MANIFEST_PATH = STAGING_DIR / "image_features_manifest.json"

compact = np.lib.format.open_memmap(
    EMBEDDINGS_PATH, mode="w+", dtype=np.float32,
    shape=(N_SUCCESS, EMBEDDING_DIM)
)

for start in tqdm(range(0, N_SUCCESS, 4096), desc="Compacting vectors"):
    end = min(start + 4096, N_SUCCESS)
    source_rows = success_indices[start:end]
    block = np.asarray(all_vectors[source_rows, :], dtype=np.float32)
    assert np.isfinite(block).all(), "Non-finite values found in a successful vector."
    norms = np.linalg.norm(block, axis=1)
    assert np.allclose(norms, 1.0, rtol=0, atol=2e-3), "A successful vector is not L2-normalized."
    compact[start:end] = block
compact.flush()
del compact

successful_rows = inventory.iloc[success_indices][["article_id", "image_path"]].reset_index(drop=True)
assert successful_rows["article_id"].is_unique
successful_rows.insert(0, "embedding_row", np.arange(N_SUCCESS, dtype=np.int64))
successful_rows["article_id"] = successful_rows["article_id"].astype("string")
successful_rows.to_parquet(IDS_PATH, index=False)

error_by_index = {}
if FAILURE_LOG_PATH.exists():
    with open(FAILURE_LOG_PATH, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                entry = json.loads(line)
                error_by_index[int(entry["index"])] = entry["error"]

failure_rows = []
for idx in failed_indices:
    failure_rows.append({
        "article_id": image_ids[idx],
        "image_path": inventory.iloc[idx]["image_path"],
        "status": int(status[idx]),
        "error": error_by_index.get(int(idx), "Failure reason unavailable"),
    })
pd.DataFrame(
    failure_rows, columns=["article_id", "image_path", "status", "error"]
).to_parquet(FAILURES_PATH, index=False)

print("Successful vectors:", N_SUCCESS)
print("Failed images:", len(failed_indices))
print("Final vector file size (MiB):", round(EMBEDDINGS_PATH.stat().st_size / 1024**2, 2))
print("Article ID dtype:", pd.read_parquet(IDS_PATH, columns=["article_id"]).dtypes["article_id"])

Compacting vectors:   0%|          | 0/26 [00:00<?, ?it/s]

Successful vectors: 105100
Failed images: 0
Final vector file size (MiB): 205.27
Article ID dtype: string


## Cell 9 — Build an exact cosine FAISS index

`IndexFlatIP` computes inner-product similarity. With L2-normalized stored and query embeddings, this is cosine similarity. Index positions are **zero-based embedding rows**, not integer article IDs.

In [13]:
embedding_matrix = np.load(EMBEDDINGS_PATH, mmap_mode="r")
assert embedding_matrix.shape == (N_SUCCESS, EMBEDDING_DIM)
assert embedding_matrix.dtype == np.float32

index = faiss.IndexFlatIP(EMBEDDING_DIM)
for start in tqdm(range(0, N_SUCCESS, 4096), desc="Building FAISS index"):
    end = min(start + 4096, N_SUCCESS)
    block = np.ascontiguousarray(embedding_matrix[start:end], dtype=np.float32)
    index.add(block)
assert index.ntotal == N_SUCCESS
faiss.write_index(index, str(INDEX_PATH))
print("Index:", type(index).__name__)
print("Index vectors:", index.ntotal)
print("Index dimensions:", index.d)
print("Index file size (MiB):", round(INDEX_PATH.stat().st_size / 1024**2, 2))

Building FAISS index:   0%|          | 0/26 [00:00<?, ?it/s]

Index: IndexFlatIP
Index vectors: 105100
Index dimensions: 512
Index file size (MiB): 205.27


## Cell 10 — Validate artifact reload, alignment and nearest-neighbor self-matches

This is a **hard publication gate**. It reloads all three artifacts from disk, checks article ID integrity and row order, checks vector norms, and checks exact vector reconstruction by FAISS for sampled rows and near-unit cosine similarity. Identical or near-identical images may tie in nearest-neighbor results, so the returned row is not required to be the queried row.

In [14]:
reloaded_vectors = np.load(EMBEDDINGS_PATH, mmap_mode="r")
reloaded_ids = pd.read_parquet(IDS_PATH)
reloaded_index = faiss.read_index(str(INDEX_PATH))

assert reloaded_vectors.dtype == np.float32
assert reloaded_vectors.shape == (N_SUCCESS, 512)
assert reloaded_index.ntotal == N_SUCCESS and reloaded_index.d == 512
assert len(reloaded_ids) == N_SUCCESS
assert reloaded_ids["embedding_row"].tolist() == list(range(N_SUCCESS)), "Mapping row indices are not contiguous."
assert reloaded_ids["article_id"].astype("string").str.fullmatch(r"[0-9]{10}").all()
assert reloaded_ids["article_id"].is_unique
assert set(reloaded_ids["article_id"]).issubset(catalog_ids)
assert reloaded_ids["article_id"].tolist() == inventory.iloc[success_indices]["article_id"].tolist()

for start in range(0, N_SUCCESS, 8192):
    block = np.asarray(reloaded_vectors[start:start + 8192], dtype=np.float32)
    assert np.isfinite(block).all(), "Non-finite values in saved array."
    assert np.allclose(np.linalg.norm(block, axis=1), 1.0, rtol=0, atol=2e-3)

sample_rows = np.unique(np.linspace(0, N_SUCCESS - 1, min(128, N_SUCCESS), dtype=np.int64))
query_vectors = np.ascontiguousarray(reloaded_vectors[sample_rows], dtype=np.float32)
scores, neighbors = reloaded_index.search(query_vectors, k=1)
reconstructed_vectors = np.stack([
    reloaded_index.reconstruct(int(row)) for row in sample_rows
]).astype(np.float32)
assert np.array_equal(reconstructed_vectors, query_vectors), "FAISS vector-to-row alignment failed."
assert np.allclose(scores[:, 0], 1.0, rtol=0, atol=2e-3), "FAISS self-similarity failed."
self_match_tie_count = int(np.count_nonzero(neighbors[:, 0] != sample_rows))

VALIDATION_PASSED = True
print("VALIDATION PASSED")
print("Embedding shape:", reloaded_vectors.shape)
print("Mapping rows:", len(reloaded_ids))
print("FAISS ntotal:", reloaded_index.ntotal)
print("FAISS vector reconstruction checks:", len(sample_rows))
print("Rank-1 nearest-neighbor ties or discrepancies:", self_match_tie_count)
print("Actual embedding coverage (% of article catalog):", round(100 * N_SUCCESS / len(catalog), 4))

VALIDATION PASSED
Embedding shape: (105100, 512)
Mapping rows: 105100
FAISS ntotal: 105100
FAISS vector reconstruction checks: 128
Rank-1 nearest-neighbor ties or discrepancies: 0
Actual embedding coverage (% of article catalog): 99.5812


## Cell 11 — Build a reproducible feature manifest

Image presence, embedding success, decode failures, and model identity are separate fields. The source-image completeness level is user-attested; the notebook does not claim independent comparison against Kaggle's full official file manifest.

In [15]:
assert VALIDATION_PASSED, "Artifact validation has not passed."

manifest = {
    "dataset": "h-and-m-personalized-fashion-recommendations",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "model_id": MODEL_ID,
    "model_revision": model_revision,
    "embedding_dim": EMBEDDING_DIM,
    "embedding_dtype": "float32",
    "normalization": "L2",
    "similarity": "cosine_via_inner_product",
    "faiss_index_type": "IndexFlatIP",
    "article_id_format": "ten_digit_string_with_leading_zeros",
    "image_mapping": "images/{article_id[:3]}/{article_id}.jpg",
    "catalog_article_count": int(len(catalog)),
    "local_image_filename_count": int(len(inventory)),
    "filename_coverage_pct": round(100 * len(inventory) / len(catalog), 4),
    "embedded_article_count": int(N_SUCCESS),
    "failed_image_count": int(len(failed_indices)),
    "embedding_catalog_coverage_pct": round(100 * N_SUCCESS / len(catalog), 4),
    "image_inventory_assurance": "user_attested_complete_local_kaggle_download",
    "independent_official_inventory_comparison": False,
    "image_decode_success_guarantee": "only_images_with_successful_clip_encoding",
    "source_fingerprint_sha256": source_fingerprint,
    "pretrained_model_only": True,
    "validation": {
        "reloaded_npy_parquet_faiss": True,
        "l2_norms_checked_all_vectors": True,
        "sample_faiss_self_matches": int(len(sample_rows)),
        "sample_vector_reconstruction_passed": True,
        "rank_1_ties_or_discrepancies": self_match_tie_count,
    },
    "versions": {
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "faiss": faiss.__version__,
        "numpy": np.__version__,
    },
    "artifact_sha256": {},
    "outputs": {
        "image_embeddings.npy": "float32[N_success, 512]",
        "image_article_ids.parquet": "embedding_row:int64, article_id:string, image_path:string",
        "image_faiss.index": "FAISS IndexFlatIP",
        "image_embedding_failures.parquet": "failed article IDs, paths, and error messages",
    },
}
for file_name in [
    "image_embeddings.npy", "image_article_ids.parquet",
    "image_faiss.index", "image_embedding_failures.parquet"
]:
    digest = hashlib.sha256()
    with open(STAGING_DIR / file_name, "rb") as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
            digest.update(block)
    manifest["artifact_sha256"][file_name] = digest.hexdigest()

MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps({
    "embedded_article_count": manifest["embedded_article_count"],
    "failed_image_count": manifest["failed_image_count"],
    "embedding_catalog_coverage_pct": manifest["embedding_catalog_coverage_pct"],
    "validation": manifest["validation"],
}, indent=2))

{
  "embedded_article_count": 105100,
  "failed_image_count": 0,
  "embedding_catalog_coverage_pct": 99.5812,
  "validation": {
    "reloaded_npy_parquet_faiss": true,
    "l2_norms_checked_all_vectors": true,
    "sample_faiss_self_matches": 128,
    "sample_vector_reconstruction_passed": true,
    "rank_1_ties_or_discrepancies": 0
  }
}


## Cell 12 — Publish verified files to Google Drive with SHA-256 checks

This is the **only** cell that creates `/content/drive/MyDrive/projects/e_commerce_recsys/data/image_features/`. It copies from validated local staging, verifies bytes with SHA-256 on both sides, and reloads the three primary files from Google Drive. The manifest is copied **last**, so its presence denotes a completed publication. Do not interrupt this cell during a file copy.

In [16]:
from google.colab import drive

drive.mount("/content/drive")
assert VALIDATION_PASSED and MANIFEST_PATH.exists(), "Run and pass local validation first."
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

# Invalidate an older publication marker before updating any assets.
old_manifest = DRIVE_DIR / "image_features_manifest.json"
if old_manifest.exists():
    old_manifest.unlink()
    print("Removed previous completion marker before publication.")

def sha256_file(path, block_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()

def publish_and_verify(source, target):
    source_hash = sha256_file(source)
    if target.is_file() and sha256_file(target) == source_hash:
        print("Already verified:", target.name)
        return
    partial = target.with_name(target.name + ".partial")
    if partial.exists():
        partial.unlink()
    shutil.copy2(source, partial)
    assert partial.stat().st_size == source.stat().st_size, f"Size mismatch: {source.name}"
    assert sha256_file(partial) == source_hash, f"Checksum mismatch: {source.name}"
    os.replace(partial, target)
    assert sha256_file(target) == source_hash, f"Final checksum mismatch: {source.name}"
    print("Published and verified:", target.name)

asset_names = [
    "image_embeddings.npy",
    "image_article_ids.parquet",
    "image_faiss.index",
    "image_embedding_failures.parquet",
]
for name in asset_names:
    publish_and_verify(STAGING_DIR / name, DRIVE_DIR / name)

# Reload the published assets, not just their local originals.
drive_vectors = np.load(DRIVE_DIR / "image_embeddings.npy", mmap_mode="r")
drive_ids = pd.read_parquet(DRIVE_DIR / "image_article_ids.parquet")
drive_index = faiss.read_index(str(DRIVE_DIR / "image_faiss.index"))
assert drive_vectors.shape == (N_SUCCESS, EMBEDDING_DIM)
assert len(drive_ids) == N_SUCCESS
assert drive_index.ntotal == N_SUCCESS and drive_index.d == EMBEDDING_DIM
assert drive_ids["article_id"].tolist() == reloaded_ids["article_id"].tolist()
for row in sample_rows:
    stored_vector = np.asarray(drive_vectors[int(row)], dtype=np.float32)
    assert np.array_equal(drive_index.reconstruct(int(row)), stored_vector)

# Publish the completion manifest last.
publish_and_verify(MANIFEST_PATH, DRIVE_DIR / "image_features_manifest.json")
print("\nGOOGLE DRIVE PUBLICATION VERIFIED:", DRIVE_DIR)
for name in asset_names + ["image_features_manifest.json"]:
    path = DRIVE_DIR / name
    print(f"  {name}: {path.stat().st_size / 1024**2:.2f} MiB")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Published and verified: image_embeddings.npy
Published and verified: image_article_ids.parquet
Published and verified: image_faiss.index
Published and verified: image_embedding_failures.parquet
Published and verified: image_features_manifest.json

GOOGLE DRIVE PUBLICATION VERIFIED: /content/drive/MyDrive/projects/e_commerce_recsys/data/image_features
  image_embeddings.npy: 205.27 MiB
  image_article_ids.parquet: 1.81 MiB
  image_faiss.index: 205.27 MiB
  image_embedding_failures.parquet: 0.00 MiB
  image_features_manifest.json: 0.00 MiB


## Cell 13 — Optional: load the published features and retrieve visually similar articles

Use this as a smoke test in a future Colab session. The original `article_id` remains a string; FAISS integer results are **matrix row indices** and must be looked up through the Parquet mapping. `IndexFlatIP` is an exact-search baseline, not a production latency optimization.

In [ ]:
# Run this cell only after the Drive publication step succeeds.
# For a new runtime, mount Google Drive before running.

vectors = np.load(DRIVE_DIR / "image_embeddings.npy", mmap_mode="r")
article_rows = pd.read_parquet(DRIVE_DIR / "image_article_ids.parquet")
faiss_index = faiss.read_index(str(DRIVE_DIR / "image_faiss.index"))

query_row = 0
query_article_id = article_rows.iloc[query_row]["article_id"]
query_vector = np.ascontiguousarray(vectors[query_row:query_row + 1], dtype=np.float32)
scores, matches = faiss_index.search(query_vector, k=6)

neighbors = pd.DataFrame({
    "rank": np.arange(1, 7),
    "article_id": article_rows.iloc[matches[0]]["article_id"].tolist(),
    "cosine_similarity": scores[0],
})
print("Query article_id:", query_article_id)
display(neighbors)
print("The first result is usually the query article; identical vectors can tie.")